# Suite SCH — Schemas

Builders, `Spec` resolution, native conversion, and on-demand `validate()` for every schema kind.
See `TestPlan.md` for the plan. Each cell is one test case, named by its ID.

In [ ]:
import { Comparison as C, Plain, Proxies, Schemas as S, Validators } from "@mbse/schemas/Framework";
import { DecodeError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, Fake, map, raises, same, text } from "./support.js";

const store = new Proxies.OfStore();

const NATIVES = [BigInt, Number, String, Boolean, Uint8Array] as const;
const bytes = (...values: number[]) => new Uint8Array(values);

## SCH-01 · `OfNative.Data` equality and validation
Supported natives validate; unsupported types and a missing type are reported. `Boolean` is not `BigInt`.

In [ ]:
for (const native of NATIVES) assert(new S.OfNative.Data(native).validate().length === 0, native.name);
for (const unsupported of [Array, Map, Object, Uint16Array, Buffer, Symbol, null]) {
  assert(new S.OfNative.Data(unsupported).validate().length > 0, String(unsupported));
}
assert(new S.OfNative.Data(String).equals(new S.OfNative.Data(String)));
assert(!new S.OfNative.Data(Boolean).equals(new S.OfNative.Data(BigInt)));
assert(new S.OfNative.Builder().create().equals(new S.OfNative.Data(null)));
assert(new S.OfNative.Builder().create().validate().length > 0);

## SCH-02 · `OfNative.resolve` accepts a type, a callable, or data

In [ ]:
assert(S.OfNative.resolve(String).equals(new S.OfNative.Data(String)));
assert(S.OfNative.resolve((n) => n.type(Uint8Array)).equals(new S.OfNative.Data(Uint8Array)));
const existing = new S.OfNative.Data(BigInt);
assert(S.OfNative.resolve(existing) === existing);
raises(TypeError, () => S.OfNative.resolve((() => undefined) as never), "must return its builder");
raises(TypeError, () => S.OfNative.resolve("str"));
raises(TypeError, () => S.OfNative.resolve(3));

## SCH-03 · `to_plain` / `from_plain`: exact types, both directions
Distinct native types are never coerced; boxed primitives and subclasses (such as `Buffer`) are rejected too. Writing
a wrong type is a program error (`TypeError`); reading one is a problem in the data (`DecodeError`, a `ValueError`,
with no location here).

In [ ]:
const cases: [unknown, unknown[]][] = [
  [BigInt, [0n, -1n, 2n ** 100n, -(2n ** 63n)]],
  [Number, [0.0, -0.0, 0.1, 1e308, 5e-324]],
  [String, ["", "x", "日本語 🚀", "\x00"]],
  [Boolean, [true, false]],
  [Uint8Array, [bytes(), bytes(0x00, 0xff), new Uint8Array(Array.from({ length: 256 }, (_, i) => i))]],
];
for (const [native, values] of cases) {
  const schema = new S.OfNative.Data(native);
  for (const value of values) {
    const back = schema.from_plain(schema.to_plain(value));
    if (value instanceof Uint8Array) assert(Object.getPrototypeOf(back) === Uint8Array.prototype && Buffer.from(back as Uint8Array).equals(Buffer.from(value)));
    else assert(Object.is(back, value), `${String(native)} ${String(value)}`);
  }
}

const wrong: [unknown, unknown[]][] = [
  [BigInt, [true, 1.0, "1", null]],
  [Number, [1n, true, null]],
  [String, [bytes(0x78), 1n, null]],
  [Boolean, [0n, 1n, "true", null]],
  [Uint8Array, [new Uint16Array([1]), "AA==", null, Buffer.from("x")]],
];
for (const [native, values] of wrong) {
  for (const value of values) raises(TypeError, () => new S.OfNative.Data(native).to_plain(value));
}

// Boxed primitives are objects, not native values: the counterpart of Python's int subclasses.
raises(TypeError, () => new S.OfNative.Data(BigInt).to_plain(Object(3n)));
raises(DecodeError, () => new S.OfNative.Data(String).from_plain(new String("x")));

for (const [native, plain, message] of [
  [BigInt, true, "expected int, got bool"],
  [BigInt, 1.0, "expected int, got float"],
  [Number, 1n, "expected float, got int"],
  [Boolean, 0n, "expected bool, got int"],
  [String, bytes(0x78), "expected str, got bytes"],
  [String, null, "expected str, got NoneType"],
  [Uint8Array, bytes(1), "expected base64 text for bytes, got bytes"],
] as const) {
  try {
    new S.OfNative.Data(native).from_plain(plain);
    throw new Error(`accepted ${message}`);
  } catch (error) {
    assert(error instanceof DecodeError && error.message === message && error.reason === message, String(error));
    assert(error.line === null && error.column === null && error.path === null);
  }
}

## SCH-04 · Bytes are strict base64 text

In [ ]:
{
  const schema = new S.OfNative.Data(Uint8Array);
  assert(schema.to_plain(bytes(0x00, 0xff)) === "AP8=");
  assert(Buffer.from(schema.from_plain(Buffer.from("hello").toString("base64")) as Uint8Array).toString() === "hello");
  for (const bad of ["AP8", "AP8=\n", "A P8=", "not base64!", "AP8=="]) raises(DecodeError, () => schema.from_plain(bad), "invalid base64 text");
}

## SCH-05 · Non-finite floats have exactly three plain forms

In [ ]:
{
  const schema = new S.OfNative.Data(Number);
  assert(schema.to_plain(Infinity) === "Infinity" && schema.to_plain(-Infinity) === "-Infinity");
  assert(schema.to_plain(NaN) === "NaN" && schema.to_plain(-NaN) === "NaN");
  assert(schema.from_plain("Infinity") === Infinity && schema.from_plain("-Infinity") === -Infinity);
  assert(Number.isNaN(schema.from_plain("NaN")));
  for (const bad of ["nan", "inf", "+Infinity", "infinity", "NAN", "1.5", ""]) {
    raises(DecodeError, () => schema.from_plain(bad), `expected a float or one of ['-Infinity', 'Infinity', 'NaN'], got ${repr(bad)}`);
  }
  raises(DecodeError, () => new S.OfNative.Data(String).from_plain(1.5), "expected str, got float"); // strings are only special for float
}

## SCH-06 · `OfAny.Builder`: selecting a kind yields that kind's data

In [ ]:
assert(new S.OfAny.Builder().as_native(String).create().constructor === S.OfNative.Data);
{
  const obj = new S.OfAny.Builder().as_object((o) => o.properties(text("x"))).create() as S.OfObject.Data;
  assert(obj.constructor === S.OfObject.Data && same([...obj.properties.keys()], ["x"]));
}
assert(new S.OfAny.Builder().as_native(String).as_native(BigInt).create().equals(new S.OfNative.Data(BigInt))); // last wins
raises(ValueError, () => new S.OfAny.Builder().create(), "no kind selected");

## SCH-07 · `OfAny.Builder` with a source: create / clone / update

In [ ]:
{
  const source = new S.OfNative.Data(String);
  raises(ValueError, () => new S.OfAny.Builder(source).as_native(BigInt).create());
  const copy = new S.OfAny.Builder(source).clone();
  assert(copy.equals(source) && copy !== source);
  assert(new S.OfAny.Builder(source).as_native(BigInt).clone().equals(new S.OfNative.Data(BigInt)) && source.equals(new S.OfNative.Data(String)));
  assert(new S.OfAny.Builder(source).as_native(Uint8Array).update() === source && source.type === Uint8Array);
  raises(TypeError, () => new S.OfAny.Builder(source).as_object((o) => o).update(), "cannot change the kind");
  raises(ValueError, () => new S.OfAny.Builder(source).update()); // nothing selected
  raises(ValueError, () => new S.OfAny.Builder().clone());
  raises(ValueError, () => new S.OfAny.Builder().as_native(String).update());
}

## SCH-08 · `OfAny.resolve` accepts value kinds only

In [ ]:
for (const data of [new S.OfNative.Data(String), new S.OfObject.Data(), new S.OfUnion.Data(), new S.OfIntersection.Data()]) {
  assert(S.OfAny.resolve(data) === data);
}
{
  const relation = new S.OfRelation.Builder().links("a", "b").create();
  raises(TypeError, () => S.OfAny.resolve(relation));
  raises(TypeError, () => new S.OfObject.Builder().properties((p) => p.name("r").of(relation as never)).create());
  raises(TypeError, () => S.OfAny.resolve(42));
}

## SCH-09 · Finalization behaves the same for every builder

In [ ]:
const builders: [new (source?: any) => any, any][] = [
  [S.OfNative.Builder, new S.OfNative.Data(String)],
  [S.OfObject.Builder, new S.OfObject.Data()],
  [S.OfRelation.Builder, new S.OfRelation.Data()],
  [S.OfAdjacency.Builder, new S.OfAdjacency.Data()],
  [S.OfUnion.Builder, new S.OfUnion.Data()],
  [S.OfIntersection.Builder, new S.OfIntersection.Data()],
  [S.OfProperty.Builder, new S.OfProperty.Data()],
];
for (const [Builder, source] of builders) {
  assert(new Builder().create().constructor === source.constructor);
  raises(ValueError, () => new Builder(source).create(), "only valid without a source");
  raises(ValueError, () => new Builder().clone(), "only valid with a source");
  raises(ValueError, () => new Builder().update(), "only valid with a source");
  const cloned = new Builder(source).clone();
  assert(cloned !== source && cloned.constructor === source.constructor);
  assert(new Builder(source).update() === source);
  raises(TypeError, () => new Builder().create("extra")); // finalizers take no arguments, as in Python
}

## SCH-10 · Builders copy their own containers, never referenced schemas

In [ ]:
const Rel = new S.OfRelation.Builder().links("a", "b").create();
const Base = new S.OfObject.Builder().properties(text("x")).relations((a) => a.name("r").of(Rel).me("a")).create();
{
  const extended = new S.OfObject.Builder(Base).properties(text("y")).clone();
  assert(same([...Base.properties.keys()], ["x"]) && same([...extended.properties.keys()], ["x", "y"]));
  assert(extended.properties !== Base.properties && extended.adjacencies !== Base.adjacencies);
  assert(extended.adjacencies.get("r") === Base.adjacencies.get("r")); // adjacency data is shared, not copied
  assert(extended.adjacencies.get("r")?.relation === Rel); // references keep identity

  const builder = new S.OfObject.Builder(Base).properties(text("z"));
  assert(!Base.properties.has("z")); // nothing reaches the source before update()
  const first = builder.clone();
  builder.properties(text("w"));
  const second = builder.clone();
  assert(!first.properties.has("w") && second.properties.has("w"));

  const same_ = new S.OfObject.Builder(Base).properties(text("v")).update();
  assert(same_ === Base && Base.properties.has("v"));
}

## SCH-11 · `OfObject`: properties and relations are keyed by name, in order

In [ ]:
{
  const o = new S.OfObject.Builder().properties(text("b"), text("a"), text("b", BigInt)).properties(text("c")).create();
  assert(same([...o.properties.keys()], ["b", "a", "c"]) && (o.properties.get("b")?.type as S.OfNative.Data).equals(new S.OfNative.Data(BigInt)));
  const Rel2 = new S.OfRelation.Builder().links("x", "y").create();
  const o2 = new S.OfObject.Builder().relations((a) => a.name("r").of(Rel).me("a"), (a) => a.name("r").of(Rel2).me("x")).create();
  assert(same([...o2.adjacencies.keys()], ["r"]) && o2.adjacencies.get("r")?.relation === Rel2);
  assert(new S.OfObject.Builder().singleton("Global.Name").create().singleton === "Global.Name");
  assert(new S.OfObject.Data().validate().length === 0);
  // Integer-like and special names keep their order, which object literals would not.
  const odd = new S.OfObject.Builder().properties(text("b"), text("2"), text("__proto__"), text("1")).create();
  assert(same([...odd.properties.keys()], ["b", "2", "__proto__", "1"]));
}

## SCH-12 · `OfObject.validate` reports clashes and nested problems

In [ ]:
{
  const clash = new S.OfObject.Builder().properties(text("r")).relations((a) => a.name("r").of(Rel).me("a")).create();
  assert(clash.validate().some((p) => p.includes("both property and adjacency")));
  const nested = new S.OfObject.Builder().properties(text("bad", Array)).create();
  assert(same(nested.validate(), ["property 'bad': unsupported native type <class 'list'>"]));
  const wrongSide = new S.OfObject.Builder().relations((a) => a.name("r").of(Rel).me("c")).create();
  assert(wrongSide.validate().some((p) => p.includes("'c' is not a link")));
  const noRelation = new S.OfObject.Builder().relations((a) => a.name("r").me("a")).create();
  assert(noRelation.validate().some((p) => p.includes("has no relation")));
}

## SCH-13 · `OfRelation.validate` covers link, property and uniqueness problems

In [ ]:
{
  const problems = (builder: S.OfRelation.Builder) => builder.create().validate();
  const R = () => new S.OfRelation.Builder();
  assert(problems(R().links("a", "b")).length === 0);
  assert(problems(R().links("a")).some((p) => p.includes("one-link")));
  assert(problems(R()).some((p) => p.includes("at least two links")));
  assert(same(problems(R().links("a", "a")), ["duplicate link names in ('a', 'a')"]));
  assert(problems(R().links("a", "b").properties(text("a"))).some((p) => p.includes("both link and property")));
  assert(problems(R().links("a", "b").unique("c")).some((p) => p.includes("unknown")));
  assert(problems(R().links("a", "b").unique()).length === 0); // unique() is trivially true
  assert(problems(R().links("a", "b").properties(text("k")).unique("a", "b", "k")).length === 0);
  assert(problems(R().links("a", "b").properties(text("k", Array))).some((p) => p.includes("property 'k'")));
  assert(same(R().links("a", "b").links("c", "d").create().links, ["c", "d"])); // links() replaces
  const r = R().links("a", "b").unique("a").unique("b").unique("a").create();
  assert(same(r.uniques.map((u) => [...u].join()), ["a", "b", "a"])); // clauses accumulate as written
}

## SCH-14 · Unions: named branches accumulate in order; validation checks count, kinds and names

In [ ]:
{
  const A = new S.OfObject.Builder().properties(text("a")).create();
  const B = new S.OfObject.Builder().properties(text("b")).create();
  const u = new S.OfUnion.Builder().branches((b) => b.name("p").of(A)).branches((b) => b.name("q").of(B)).create();
  assert(same(u.branches.map((b) => b.type), [A, B]) && same(u.branches.map((b) => b.name), ["p", "q"]));
  assert(same([...u.properties.keys()], ["p", "q"]) && same([...u.properties.values()].map((b) => b.type), [A, B]));
  assert(u.branches[0] instanceof S.OfUnion.Branch);
  assert(u.validate().length === 0);
  assert(same(new S.OfUnion.Builder().branches((b) => b.name("p").of(A)).create().validate(),
    ["a union needs at least two branches"]));
  const native = (t: S.OfAny.Builder) => t.as_native(String);
  assert(same(new S.OfUnion.Builder().branches((b) => b.name("p").of(A), (b) => b.name("q").of(native)).create().validate(),
    ["union branches must all be the same kind"]));
  raises(TypeError, () => new S.OfUnion.Builder().branches((b) => b.of(String as never)), "as_native(String)"); // a class is not a Spec
  raises(TypeError, () => new S.OfObject.Builder().properties((() => "not a builder") as never), "must return its builder");
  const unnamed = new S.OfUnion.Builder().branches((b) => b.of(A), (b) => b.name("q").of(B)).create().validate();
  assert(same(unnamed, ["branch 0 has no name"]));
  const twice = new S.OfUnion.Builder().branches((b) => b.name("p").of(A), (b) => b.name("p").of(B)).create().validate();
  assert(same(twice, ["branch name 'p' is used more than once"]));
}

## SCH-15 · Intersections: named parts accumulate; parts may declare the same property with different types

In [ ]:
{
  const X1 = new S.OfObject.Builder().properties(text("x")).create();
  const X2 = new S.OfObject.Builder().properties((p) => p.name("x").of((t) => t.as_native(String)), text("y")).create();
  const X3 = new S.OfObject.Builder().properties(text("x", BigInt)).create();
  type Part = Parameters<S.OfIntersection.Builder["parts"]>[0];
  const x1: Part = (p) => p.name("x1").of(X1), x2: Part = (p) => p.name("x2").of(X2), x3: Part = (p) => p.name("x3").of(X3);
  const i = new S.OfIntersection.Builder().parts(x1).parts(x2).create();
  assert(same(i.parts.map((p) => p.type), [X1, X2]) && same([...i.properties.keys()], ["x1", "x2"]));
  assert(i.parts[0] instanceof S.OfIntersection.Part);
  assert(new S.OfIntersection.Builder().parts(x1, x2, x3).create().validate().length === 0); // each part keeps its own 'x'
  assert(same(new S.OfIntersection.Builder().parts(x1).create().validate(), ["an intersection needs at least two parts"]));
  assert(same(new S.OfIntersection.Builder().parts(x1, (p) => p.name("s").of(new S.OfNative.Data(String))).create().validate(),
    ["intersection parts must all be the same kind"]));
  assert(same(new S.OfIntersection.Builder().parts(x1, (p) => p.of(X2), x1).create().validate(),
    ["part 1 has no name", "part name 'x1' is used more than once"]));
}

## SCH-16 · Finding F2 (pinned): `validate()` accepts names that bindings may reserve
Empty names, names with spaces or `$`, leading underscores, names of builder methods, and JavaScript's own object
members all validate. Whether the schema layer should reject them is an open question in FRAMEWORK.md.

In [ ]:
{
  const names = ["", "with space", "$ref", "_values", "create", "accept", "constructor", "toString", "__proto__", "then"];
  const odd = new S.OfObject.Builder().properties(...names.map((n) => text(n))).create();
  assert(same(odd.validate(), ["name '$ref' is reserved: names starting with '$' belong to the wire format"])); // the others: F2
  assert(same([...odd.properties.keys()], names));
}

## SCH-17 · Union and intersection selection; identity equality; Spec errors name what they got

In [ ]:
{
  const A = new S.OfObject.Builder().properties(text("a")).create();
  const B = new S.OfObject.Builder().properties(text("b")).create();
  const X1 = new S.OfObject.Builder().properties(text("x")).create();
  const X2 = new S.OfObject.Builder().properties((p) => p.name("x").of((t) => t.as_native(String)), text("y")).create();
  const U = new S.OfAny.Builder().as_union((u) => u.branches((b) => b.name("p").of(A), (b) => b.name("q").of(B))).create() as S.OfUnion.Data;
  const I = new S.OfAny.Builder().as_intersection((i) => i.parts((p) => p.name("x1").of(X1), (p) => p.name("x2").of(X2))).create() as S.OfIntersection.Data;
  assert(U.constructor === S.OfUnion.Data && U.branches.length === 2 && I.constructor === S.OfIntersection.Data && I.parts.length === 2);
  assert(S.OfUnion.resolve(U) === U && S.OfIntersection.resolve(I) === I);
  assert(S.OfUnion.resolve((u) => u).constructor === S.OfUnion.Data && S.OfIntersection.resolve((i) => i).constructor === S.OfIntersection.Data);
  for (const Data of [S.OfObject.Data, S.OfRelation.Data, S.OfAdjacency.Data, S.OfUnion.Data, S.OfIntersection.Data, S.OfProperty.Data,
    S.OfUnion.Branch]) {
    const data = new Data();
    assert(data.equals(data) && !data.equals(new Data())); // schemas other than natives compare by identity
  }
  assert(same(new S.OfObject.Data({ properties: new Map([["x", new S.OfProperty.Data({ name: "x", type: "junk" as never })]]) }).validate(), ["property 'x': not a schema: 'junk'"]));
  const untyped = new S.OfUnion.Builder().branches((b) => b.name("p"), (b) => b.name("q").of(A)).create();
  assert(untyped.branches[0]?.type === null && untyped.branches[0]?.equals(untyped.branches[0]));
  assert(untyped.validate().includes("union branches must all be the same kind"));

  class Custom {}
  raises(TypeError, () => S.OfObject.resolve(Custom), "a class is not a Spec here");
  raises(TypeError, () => new S.OfObject.Builder().properties((() => "not a builder") as never), "must return its builder, got 'not a builder'");
  raises(TypeError, () => new S.OfObject.Builder().properties((() => ({})) as never), "must return its builder, got <object object");
  raises(TypeError, () => new S.OfObject.Builder().properties((() => () => null) as never), "must return its builder, got <function");
}

## SCH-18 · Native types are tokens, with an optional width in bits or bytes
A host type is shorthand for the `basic` token of the same name. An implementation reads `basic` tokens and its own
format's (`typescript5`); a token in another format is valid, but describes values this implementation cannot convert,
except a `python3` token, which names only the basic types.

In [ ]:
{
  const N = S.OfNative;
  const int = new N.Data(BigInt);
  assert(int.token instanceof N.Token && int.token.equals(new N.Token("basic", "int")) && int.type === BigInt && int.host() === BigInt);
  assert((N.resolve((n) => n.type(Uint8Array)).token as S.OfNative.Token).equals(new N.Token("basic", "bytes")));
  const own = N.resolve((n) => n.token("typescript5", "BigInt"));
  assert(own.type === BigInt && own.validate().length === 0 && !own.equals(int)); // the same host type, another token
  assert(N.resolve((n) => n.token("python3", "int")).type === null); // the other implementation's format
  const wide = N.resolve((n) => n.type(BigInt).bytes(4n));
  assert(wide.equals(new N.Data(BigInt, { bytes: 4n })) && !wide.equals(int) && wide.bits === null && wide.bytes === 4n);
  assert(wide.validate().length === 0 && wide.to_plain(1n) === 1n); // a width is size only: values are not checked against it
  const foreign = N.resolve((n) => n.token("ccpp", "int32_t").bits(32n));
  assert(String(foreign.token) === "the ccpp type 'int32_t'" && foreign.type === null && foreign.validate().length === 0);
  for (const call of [() => foreign.host(), () => foreign.to_plain(1n)]) {
    raises(TypeError, call, "the ccpp type 'int32_t' has no type in this implementation");
  }
  raises(DecodeError, () => foreign.from_plain(1n), "the ccpp type 'int32_t' has no type in this implementation");
  assert(same(N.resolve((n) => n.token("basic", "long")).validate(), ["basic has no type 'long'"]));
  assert(same(N.resolve((n) => n.token("python3", "decimal.Decimal")).validate(), ["python3 has no type 'decimal.Decimal'"])); // only the basic types
  assert(N.resolve((n) => n.token("python3", "int")).validate().length === 0);
  assert(same(N.resolve((n) => n.token("", "x")).validate(), ["a token needs a format and a name"]));
  assert(same(N.resolve((n) => n.type(BigInt).bits(0n).bytes(true as never)).validate(), [
    "a width in bits must be a positive int or a term, got 0", "a width in bytes must be a positive int or a term, got True",
    "a width is in bits or in bytes, not both"]));

  const Wire = new S.OfObject.Builder().name("Wire").ref().properties((p) => p.name("word").of(foreign)).create();
  store.register(Wire);
  assert(same(Validators.Validate(store)(Wire, new Fake("Wire", { word: 1n })), [
    "Wire#0.word: the ccpp type 'int32_t' has no type in this implementation"]));
  raises(TypeError, () => new C.OfNative(foreign, 1n), "the ccpp type 'int32_t' has no type in this implementation");
  assert(new C.OfNative(own, 1n).compare(new C.OfNative(int, 1n)) === 0); // both hold a bigint
  assert(S.isNativeOf(Array, []) && !S.isNativeOf(Array, null) && !S.isNativeOf(Array, undefined)); // other classes too
}

## SCH-19 · Reference object schemas: marked with `.ref()`, never a property's type, the only snapshot roots
A singleton's schema is a reference object schema. Only reference objects are built with a store's builders, written
as a snapshot's root, or linked in a snapshot; a value object is built through its owner.

In [ ]:
{
  const Thing = new S.OfObject.Builder().name("Thing").ref().properties(text("name")).create();
  const Part = new S.OfObject.Builder().name("Part").properties(text("name")).create();
  assert(Thing.ref && !Part.ref && Thing.validate().length === 0 && Part.validate().length === 0);
  assert(new S.OfObject.Builder().singleton("Only").create().ref);
  assert(same(new S.OfObject.Data({ singleton: "Only" }).validate(), ["a singleton's schema must be a reference object schema"]));
  const held = "a reference object schema cannot be a property's type";
  assert(same(new S.OfObject.Builder().properties((p) => p.name("thing").of(Thing)).create().validate(), [`property 'thing': ${held}`]));
  const Either = new S.OfUnion.Builder().branches((b) => b.name("thing").of(Thing), (b) => b.name("part").of(Part)).create();
  const Both = new S.OfIntersection.Builder().parts((p) => p.name("thing").of(Thing), (p) => p.name("part").of(Part)).create();
  for (const holder of [new S.OfObject.Builder(), new S.OfRelation.Builder().links("a", "b")]) {
    assert(same(holder.properties((p) => p.name("e").of(Either), (p) => p.name("i").of(Both)).create().validate(), [
      `property 'e': ${held}`, `property 'i': ${held}`]));
  }

  store.register(Part);
  raises(TypeError, () => (store as any).Part(), "'Part' is a value object schema; a value object is built through its owner");
  raises(TypeError, () => Plain.ToPlain(store)(Part, new Fake("Part", { name: "p" })),
    "a snapshot's root must be a reference object; 'Part' is a value object schema");
  raises(TypeError, () => Plain.FromPlain(store)(Part, map({ root: "s0", objects: { s0: {} } })),
    "the root schema must be a reference object schema");
  store.register(Thing);
  const Pair = new S.OfRelation.Builder().links("thing", "part").create();
  new S.OfObject.Builder(Thing).relations((r) => r.name("parts").of(Pair).me("thing")).update();
  const linked = map({ root: "s0", objects: { s0: { parts: [{ part: { $ref: "s1", $schema: "Part" } }] }, s1: {} } });
  raises(DecodeError, () => Plain.FromPlain(store)(Thing, linked), "$.objects.s1: 'Part' is not a reference object schema");
}

## SCH-20 · Every kind of schema may be named: identifiers separated by dots

In [ ]:
{
  const named = [S.OfNative.resolve((n) => n.name("units.Meters").type(Number)),
    new S.OfObject.Builder().name("crm.Contact").ref().create(),
    new S.OfRelation.Builder().name("crm.Knows").links("a", "b").create(),
    new S.OfUnion.Builder().name("crm.Reach").branches(text("a"), text("b")).create(),
    new S.OfIntersection.Builder().name("crm.Stamped").parts(text("a"), text("b")).create(),
    new S.OfIndexed.Builder().name("crm.Tags").of((t) => t.as_native(String)).create()];
  assert(same(named.map((s) => s.name), ["units.Meters", "crm.Contact", "crm.Knows", "crm.Reach", "crm.Stamped", "crm.Tags"]));
  assert(named.every((s) => s.validate().length === 0) && new S.OfObject.Builder().create().name === null); // names are optional
  for (const bad of ["", "crm.", ".Contact", "crm..Contact", "crm-Contact", "1crm.Contact", "crm.Contact v2"]) {
    assert(same(new S.OfObject.Builder().name(bad).create().validate(),
      [`name ${repr(bad)} is not identifiers separated by dots, e.g. 'crm.Contact'`]), bad);
  }
  assert(!named[0]!.equals(new S.OfNative.Data(Number))); // a native's name is part of it
  const renamed = new S.OfObject.Builder(named[1] as S.OfObject.Data).name("crm.Person").clone();
  assert(renamed.name === "crm.Person" && named[1]!.name === "crm.Contact" && renamed.ref);
}

## SCH-21 · Every element may be described: schemas, properties, adjacencies and members

In [ ]:
{
  const Phone = new S.OfObject.Builder().name("crm.Phone").ref().description("A number to call").properties(
    (p) => p.name("number").of((t) => t.as_native(String)).description("Digits only")).create();
  const Calls = new S.OfRelation.Builder().name("crm.Calls").description("Who called whom").links("caller", "callee").properties(
    (p) => p.name("at").of((t) => t.as_native(String)).description("When")).create();
  new S.OfObject.Builder(Phone).relations((a) => a.name("calls").of(Calls).me("caller").description("Calls made")).update();
  const Reach = new S.OfUnion.Builder().description("One way to reach").branches(
    (b) => b.name("phone").of((t) => t.as_native(String)).description("By phone"), text("mail")).create();
  const Both = new S.OfIntersection.Builder().description("Both at once").parts(
    (p) => p.name("a").of((t) => t.as_native(String)).description("The first"), text("b")).create();
  const Tags = new S.OfIndexed.Builder().description("Any number").of((t) => t.as_native(String)).create();
  const Meters = S.OfNative.resolve((n) => n.type(Number).description("A length"));
  const described = [Meters, Phone, Calls, Reach, Both, Tags];
  assert(same(described.map((s) => s.description), [
    "A length", "A number to call", "Who called whom", "One way to reach", "Both at once", "Any number"]));
  assert(described.every((s) => s.validate().length === 0) && new S.OfObject.Builder().create().description === null); // optional
  assert(Phone.properties.get("number")?.description === "Digits only" && Calls.properties.get("at")?.description === "When");
  assert(Phone.adjacencies.get("calls")?.description === "Calls made");
  assert(Reach.properties.get("phone")?.description === "By phone" && Reach.properties.get("mail")?.description === null);
  assert(Both.properties.get("a")?.description === "The first");
  assert(!Meters.equals(new S.OfNative.Data(Number)) && Meters.equals(new S.OfNative.Data(Number, { description: "A length" }))); // part of a native
  const redescribed = new S.OfIndexed.Builder(Tags).description("Labels").clone();
  assert(redescribed.description === "Labels" && Tags.description === "Any number");
  // A description is text; anything else is reported where it is
  const ofStr = (t: any) => t.as_native(String);
  assert(same(new S.OfObject.Builder().description(true as never).create().validate(), ["a description is text, got bool"]));
  assert(same(new S.OfObject.Builder().properties((p) => p.name("p").of(ofStr).description(true as never)).create().validate(), [
    "property 'p': a description is text, got bool"]));
  assert(same(new S.OfRelation.Builder().links("a", "b").properties(
    (p) => p.name("p").of(ofStr).description(true as never)).create().validate(), ["property 'p': a description is text, got bool"]));
  assert(same(new S.OfObject.Builder().relations((a) => a.name("calls").of(Calls).me("caller").description(true as never)).create(
    ).validate(), ["adjacency 'calls': a description is text, got bool"]));
  assert(same(new S.OfUnion.Builder().branches((b) => b.name("x").of(ofStr).description(true as never), text("y")).create().validate(), [
    "branch 'x': a description is text, got bool"]));
  assert(same(new S.OfIntersection.Builder().parts(text("x"), (p) => p.name("y").of(ofStr).description(true as never)).create().validate(
    ), ["part 'y': a description is text, got bool"]));
  assert(same(new S.OfNative.Data(Number, { description: true as never }).validate(), ["a description is text, got bool"]));
  assert(same(new S.OfIndexed.Builder().description(true as never).of(ofStr).create().validate(), ["a description is text, got bool"]));
}